# RandomForest агрономические модели — демо

Загружаем готовые артефакты из `../models/` и демонстрируем работу 4 моделей.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from pathlib import Path

warnings.filterwarnings('ignore')

MODELS_DIR = Path('..') / 'models'

# Crop
model_crop = joblib.load(MODELS_DIR / 'crop_model.pkl')
le_crop    = joblib.load(MODELS_DIR / 'crop_label_encoder.pkl')
# Fertilizer
model_fert    = joblib.load(MODELS_DIR / 'fertilizer_model.pkl')
fert_encoders = joblib.load(MODELS_DIR / 'fertilizer_encoders.pkl')
# Pesticide
model_pest    = joblib.load(MODELS_DIR / 'pesticide_model.pkl')
pest_encoders = joblib.load(MODELS_DIR / 'pesticide_encoders.pkl')
# Disease
model_disease = joblib.load(MODELS_DIR / 'disease_model.pkl')
model_risk    = joblib.load(MODELS_DIR / 'disease_risk_model.pkl')
disease_enc   = joblib.load(MODELS_DIR / 'disease_encoders.pkl')

print('Все модели загружены')

## 1. Crop Recommender — рекомендация культуры

Признаки: N, P, K, temperature, humidity, ph, rainfall

In [ ]:
print(f'Деревьев: {model_crop.n_estimators}')
print(f'Классов : {len(le_crop.classes_)}')
print(f'Классы  : {list(le_crop.classes_)}')

feat_names = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
imp = pd.Series(model_crop.feature_importances_, index=feat_names).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(7, 4))
imp.plot(kind='barh', ax=ax, color='steelblue', alpha=0.85)
ax.set_title('Важность признаков (crop model, Gini)')
plt.tight_layout()
plt.show()

In [ ]:
examples = [
    {'label': 'Омская обл.', 'N': 50, 'P': 35, 'K': 120,
     'temperature': 19.5, 'humidity': 62.0, 'ph': 6.9, 'rainfall': 210.0},
    {'label': 'Алтай (засуха)', 'N': 30, 'P': 20, 'K': 80,
     'temperature': 23.0, 'humidity': 45.0, 'ph': 7.2, 'rainfall': 130.0},
]
for ex in examples:
    X = pd.DataFrame([{k: v for k, v in ex.items() if k != 'label'}])
    probs = model_crop.predict_proba(X)[0]
    top = np.argsort(probs)[::-1]
    print(f'\n>>> {ex["label"]}')
    for i in top[:3]:
        print(f'  {le_crop.classes_[i]:20} {probs[i]:.1%}')

## 2. Fertilizer Recommender — рекомендация удобрения

Признаки: crop_type, soil_type, deficiency_level

In [ ]:
def safe_encode(le, val):
    val = str(val)
    if val not in set(le.classes_):
        val = le.classes_[0]
    return int(le.transform([val])[0])

le_fert = fert_encoders['fertilizer']
print(f'Удобрений: {len(le_fert.classes_)}')
print(f'Классы  : {list(le_fert.classes_)}')

def recommend_fertilizer(crop_type, soil_type, deficiency_level):
    X = np.array([[
        safe_encode(fert_encoders['crop_type'], crop_type),
        safe_encode(fert_encoders['soil_type'], soil_type),
        safe_encode(fert_encoders['deficiency_level'], deficiency_level),
    ]])
    probs = model_fert.predict_proba(X)[0]
    top = np.argsort(probs)[::-1]
    print(f'\n  {crop_type} | {soil_type} | {deficiency_level}')
    for i in top[:3]:
        print(f'    {le_fert.classes_[i]:30} {probs[i]:.1%}')

recommend_fertilizer('wheat', 'chernozem', 'Nitrogen')
recommend_fertilizer('sunflower', 'loamy', 'Phosphorus')
recommend_fertilizer('corn', 'solonetz', 'Potassium')

## 3. Pesticide Recommender — рекомендация пестицида

Признаки: crop, pest_type, intensity

In [ ]:
le_pest = pest_encoders['pesticide']
print(f'Пестицидов: {len(le_pest.classes_)}')
print(f'Классы   : {list(le_pest.classes_)}')

def recommend_pesticide(crop, pest_type, intensity):
    X = np.array([[
        safe_encode(pest_encoders['crop'], crop),
        safe_encode(pest_encoders['pest_type'], pest_type),
        safe_encode(pest_encoders['intensity'], intensity),
    ]])
    probs = model_pest.predict_proba(X)[0]
    top = np.argsort(probs)[::-1]
    print(f'\n  {crop} | {pest_type} | {intensity}')
    for i in top[:3]:
        print(f'    {le_pest.classes_[i]:30} {probs[i]:.1%}')

recommend_pesticide('wheat', 'fungal', 'high')
recommend_pesticide('corn', 'insect', 'medium')
recommend_pesticide('sunflower', 'weed', 'low')

## 4. Disease Predictor — прогноз болезни и риска

Признаки: crop, growth_stage + temperature, humidity, rainfall
Таргеты: disease (болезнь) + risk_level (low / medium / high)

In [ ]:
print('DiseaseClassifier:')
print(f'  Деревьев: {model_disease.n_estimators}')
print(f'  Болезней: {len(disease_enc["disease"].classes_)}')
print(f'  Классы  : {list(disease_enc["disease"].classes_)}')
print()
print('RiskClassifier:')
print(f'  Деревьев: {model_risk.n_estimators}')
print(f'  Уровни  : {list(disease_enc["risk_level"].classes_)}')

dis_imp = pd.Series(
    model_disease.feature_importances_,
    index=['crop', 'growth_stage', 'temperature', 'humidity', 'rainfall']
).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(7, 3))
dis_imp.plot(kind='barh', ax=ax, color='steelblue', alpha=0.85)
ax.set_title('Важность признаков (disease model, Gini)')
plt.tight_layout()
plt.show()

In [ ]:
def predict_disease(crop, growth_stage, temperature, humidity, rainfall):
    X = np.array([[
        safe_encode(disease_enc['crop'], crop),
        safe_encode(disease_enc['growth_stage'], growth_stage),
        temperature, humidity, rainfall
    ]])
    d_probs = model_disease.predict_proba(X)[0]
    r_probs = model_risk.predict_proba(X)[0]
    top = np.argsort(d_probs)[::-1]
    risk = disease_enc['risk_level'].classes_[np.argmax(r_probs)]
    print(f'\n  {crop} | {growth_stage} | T={temperature} H={humidity}% rain={rainfall}mm')
    print(f'  Риск: {risk} ({r_probs.max():.1%})')
    for i in top[:3]:
        print(f'    {disease_enc["disease"].classes_[i]:35} {d_probs[i]:.1%}')

predict_disease('spring_wheat', 'flowering', 22.0, 84.0, 95.0)
predict_disease('corn', 'vegetative', 26.0, 55.0, 40.0)
predict_disease('sunflower', 'ripening', 18.0, 78.0, 60.0)